# Hour 2 · Keywords and TF-IDF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/2a_tfidf_keywords.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F2a_tfidf_keywords.ipynb)


**Goal:** automatically surface the vocabulary *characteristic* of each tablet and genre.

*Reading:* `docs/04-genres.md`.

## 0. Setup


In [ ]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

## 1. Pick a genre-diverse working set

In [ ]:
sample = [t for t in texts if t["genre"] in {"myth","letter","ritual","divination"}
          and len(t["tokens"]) >= 25]
from collections import Counter
print(len(sample), "tablets:", Counter(t["genre"] for t in sample))

## 2. Plain word frequencies first

In [ ]:
from data.loader import token_counts
token_counts(sample).most_common(15)

## 3. TF-IDF — distinctive words per tablet

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from data.loader import corpus_as_documents
labels, docs = corpus_as_documents(sample)
vec = TfidfVectorizer(token_pattern=r"[^\s]+")
X = vec.fit_transform(docs)
print("matrix (tablets x vocab):", X.shape)

## 4. Top keywords per tablet — can you guess the genre?

In [ ]:
import numpy as np
feat = np.array(vec.get_feature_names_out())
def top_kw(i, n=6):
    row = X[i].toarray().ravel()
    idx = row.argsort()[::-1][:n]
    return [feat[j] for j in idx if row[j] > 0]
for i, t in enumerate(sample[:15]):
    print(f'{t["ktu"]:7s} [{t["genre"]:11s}] {top_kw(i)}')

## 5. Discussion
- How much of the genre is visible from keywords alone?
- **Caveats:** word *forms* vs lemmas (CUC has no lemmatisation — homographs blur the signal); short/damaged tablets; genre formulas dominating.

> **Production version:** UgaritLab (`generate_stats.py`) computes TF-IDF over the whole corpus and projects it for browsing.

## ✍️ Your turn
Edit one value below and re-run. Nothing here can break the notebook — if it goes sideways, just re-run the setup cell.


In [ ]:
# Show more or fewer keywords per tablet, and look at a different slice.
N = 10                       # ← number of keywords
START = 15                   # ← which tablets to inspect (try 0, 30, 45)
for i in range(START, min(START+12, len(sample))):
    print(f'{sample[i]["ktu"]:7s} [{sample[i]["genre"]:11s}] {top_kw(i, N)}')
# Hint: longer keyword lists start to include shared particles — where does the signal fade?